In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [2]:
con = sqlite3.connect("../chembl_36/chembl_36_sqlite/chembl_36.db")

In [3]:
cols = "standard_value, standard_type, pchembl_value, mw_freebase, alogp, hba, hbd, psa, rtb, ro3_pass, num_ro5_violations, full_mwt, aromatic_rings, heavy_atoms, qed_weighted, np_likeness_score, bei, sei, le, lle, max_phase, structure_type, chirality"



In [4]:
df = pd.read_sql_query("""SELECT """ + cols + """ FROM activities 
                    JOIN compound_properties ON activities.molregno == compound_properties.molregno
                    JOIN ligand_eff ON activities.activity_id == ligand_eff.activity_id
                    JOIN molecule_dictionary ON activities.molregno == molecule_dictionary.molregno
                    JOIN assays ON activities.assay_id == assays.assay_id""", con)

In [5]:
df.tail(100)

,standard_value,standard_type,pchembl_value,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,...,heavy_atoms,qed_weighted,np_likeness_score,bei,sei,le,lle,max_phase,structure_type,chirality
2191325,8.5,IC50,8.07,361.37,1.73,5.0,2.0,103.54,6.0,N,...,26.0,0.82,-0.43,22.33,7.79,0.42,6.34,NaN,MOL,-1
2191326,1.2,IC50,8.92,361.37,1.58,5.0,2.0,103.54,6.0,N,...,26.0,0.81,-0.25,24.69,8.62,0.47,7.34,NaN,MOL,-1
2191327,3789.0,IC50,5.42,341.37,1.25,5.0,2.0,103.54,6.0,N,...,25.0,0.82,0.00,15.88,5.24,0.30,4.17,NaN,MOL,-1
2191328,11.0,IC50,7.96,373.38,1.73,5.0,2.0,103.54,6.0,N,...,27.0,0.80,0.03,21.32,7.69,0.40,6.23,NaN,MOL,-1
2191329,11.0,IC50,7.96,341.37,1.25,5.0,2.0,103.54,6.0,N,...,25.0,0.82,-0.06,23.31,7.69,0.43,6.71,NaN,MOL,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2191420,0.3,IC50,9.52,365.34,1.14,5.0,2.0,103.54,6.0,N,...,26.0,0.80,-0.09,26.07,9.20,0.50,8.38,NaN,MOL,-1
2191421,740.0,IC50,6.13,435.48,2.54,7.0,2.0,129.32,7.0,N,...,32.0,0.58,-0.68,14.08,4.74,0.26,3.59,NaN,MOL,-1
2191422,72.0,IC50,7.14,455.90,2.89,7.0,2.0,129.32,7.0,N,...,32.0,0.56,-0.62,15.67,5.52,0.30,4.25,NaN,MOL,-1
2191423,59.0,IC50,7.23,436.47,2.13,6.0,3.0,136.40,7.0,N,...,32.0,0.52,-0.37,16.56,5.30,0.31,5.10,NaN,MOL,-1


In [6]:
df = df.dropna(axis=1, how='all')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2191425 entries, 0 to 2191424
Data columns (total 23 columns):
 #   Column              Dtype  
---  ------              -----  
 0   standard_value      float64
 1   standard_type       object 
 2   pchembl_value       float64
 3   mw_freebase         float64
 4   alogp               float64
 5   hba                 float64
 6   hbd                 float64
 7   psa                 float64
 8   rtb                 float64
 9   ro3_pass            object 
 10  num_ro5_violations  float64
 11  full_mwt            float64
 12  aromatic_rings      float64
 13  heavy_atoms         float64
 14  qed_weighted        float64
 15  np_likeness_score   float64
 16  bei                 float64
 17  sei                 float64
 18  le                  float64
 19  lle                 float64
 20  max_phase           float64
 21  structure_type      object 
 22  chirality           int64  
dtypes: float64(19), int64(1), object(3)
memory usage: 384.5+ 

In [8]:
df = df[df['structure_type'] == 'MOL']
df = df.drop("structure_type", axis=1)

In [9]:
df = df[df['max_phase'] != -1]
df['clinical_trial'] = df['max_phase'].notna() 
df = df.drop("max_phase", axis=1)

In [10]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2129486 entries, 0 to 2191424
Data columns (total 22 columns):
 #   Column              Dtype  
---  ------              -----  
 0   standard_value      float64
 1   standard_type       object 
 2   pchembl_value       float64
 3   mw_freebase         float64
 4   alogp               float64
 5   hba                 float64
 6   hbd                 float64
 7   psa                 float64
 8   rtb                 float64
 9   ro3_pass            object 
 10  num_ro5_violations  float64
 11  full_mwt            float64
 12  aromatic_rings      float64
 13  heavy_atoms         float64
 14  qed_weighted        float64
 15  np_likeness_score   float64
 16  bei                 float64
 17  sei                 float64
 18  le                  float64
 19  lle                 float64
 20  chirality           int64  
 21  clinical_trial      bool   
dtypes: bool(1), float64(18), int64(1), object(2)
memory usage: 359.5+ MB


In [11]:
df.to_csv("../chembl_36/chembl_36_cleaned.csv")

In [12]:
## Get a balanced dataset because 
df['clinical_trial'].value_counts()
# Yikes

clinical_trial
False    2021522
True      107964
Name: count, dtype: int64

In [13]:
df_clinical_trial_false = df[df['clinical_trial'] == False]
df_clinical_trial_true = df[df['clinical_trial'] == True]

In [14]:
temp = df_clinical_trial_false.sample(n=107964)

df_balanced = pd.concat([temp, df_clinical_trial_true], axis=0)
df_balanced = df_balanced.sample(frac=1) # shuffle the data just in case
df_balanced

,standard_value,standard_type,pchembl_value,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,...,aromatic_rings,heavy_atoms,qed_weighted,np_likeness_score,bei,sei,le,lle,chirality,clinical_trial
1592107,10.40,IC50,7.98,524.69,4.82,8.0,3.0,108.48,10.0,N,...,3.0,37.0,0.35,-1.71,15.21,7.36,0.29,3.16,2,True
1654867,250.00,IC50,6.60,354.45,3.21,4.0,2.0,76.02,8.0,N,...,2.0,26.0,0.71,-1.01,18.63,8.68,0.35,3.39,-1,False
1169740,6.40,IC50,8.19,554.54,5.97,6.0,2.0,85.25,9.0,N,...,4.0,40.0,0.20,-1.46,14.78,9.61,0.28,2.22,-1,False
1856956,897.00,IC50,6.05,356.25,3.54,5.0,0.0,52.83,4.0,N,...,3.0,25.0,0.67,-2.36,16.97,11.45,0.33,2.51,-1,False
658644,16.00,Ki,7.80,304.32,-1.82,7.0,5.0,142.11,3.0,N,...,1.0,20.0,0.44,-0.01,25.62,5.49,0.53,9.62,-1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1178229,11.00,IC50,7.96,426.52,4.09,4.0,1.0,82.01,4.0,N,...,3.0,32.0,0.68,-0.84,18.66,9.70,0.34,3.87,-1,False
140599,70.00,Ki,7.16,531.50,4.94,3.0,2.0,93.14,7.0,N,...,3.0,33.0,0.42,-1.14,13.46,7.68,0.30,2.21,-1,False
185165,4300.00,IC50,5.37,270.24,2.58,5.0,3.0,90.90,1.0,N,...,3.0,20.0,0.59,1.27,19.86,5.90,0.37,2.79,2,True
952481,60.00,IC50,7.22,393.45,1.58,9.0,4.0,149.66,6.0,N,...,3.0,29.0,0.49,-1.41,18.36,4.83,0.34,5.64,-1,False


In [15]:
df_balanced.to_csv("../chembl_36/chembl_36_cleaned_balanced.csv")